In [7]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from itertools import product
import sys
sys.path.append('../')
from model_classes import GAT
from helper_functions import create_data, train_one_epoch_GAT, evaluate_GAT, predict_GAT, compute_metrics

In [8]:
adj_matrix = torch.tensor(np.load('../data_folder/Adjacency_matrix.npy'), dtype=torch.float32)
open_prices = pd.read_csv('../data_folder/open_prices_interp.csv', index_col=0)

x = open_prices.values

X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(
    x, batch_size=32, flatten_data=False, flatten_time_features=True
)

torch.Size([984, 460, 40])
torch.Size([984, 460])


In [9]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
search_space = {
    'c_hidden': [16, 32],
    'num_heads': [2, 4],
    'alpha': [0.2, 0.4],
    'lr': [1e-3, 1e-4],
}
num_epochs = 8
patience = 2

input_dim = X_train.shape[-1]  # 40
num_relations = adj_matrix.shape[-1]  # 3

keys = list(search_space.keys())
param_grid = [dict(zip(keys, values)) for values in product(*(search_space[k] for k in keys))]
print(f'Running {len(param_grid)} trials')

Running 16 trials


In [10]:
results = []
best = {'best_val_mse': float('inf')} #tracks the single best model seen so far 

for trial_idx, params in enumerate(param_grid, start=1): #loops through all hyperparameter combinations
    
    # num_heads must divide c_hidden cleanly
    if params['c_hidden'] % params['num_heads'] != 0:
        continue

    torch.manual_seed(seed)
    np.random.seed(seed)
    #create a fresh GAT with this trial's hyperparameters
    model = GAT(    
        c_in=input_dim,
        c_hidden=params['c_hidden'],
        c_out=1,
        num_relations=num_relations,
        num_heads=params['num_heads'],
        alpha=params['alpha']
    )

    optimizer = optim.Adam(model.parameters(), lr=params['lr'])
    criterion = nn.MSELoss()

    best_val_mse = float('inf')
    best_state_dict = None
    patience_counter = 0

    print(f"Trial {trial_idx:02d}/{len(param_grid)} | {params}")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch_GAT(model, train_loader, adj_matrix, optimizer, criterion)
        val_mse = evaluate_GAT(model, val_loader, adj_matrix, criterion)

        print(f"  Epoch {epoch:02d} | train loss {train_loss:.6f} | val MSE {val_mse:.6f}")

        #if val MSE improves we save the model weights and reset the counter. if it gets worse 2 times in a row we stop early
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state_dict = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  Early stopping")
                break

    results.append({**params, 'best_val_mse': best_val_mse})

    if best_val_mse < best['best_val_mse']:
        best = {
            'best_val_mse': best_val_mse,
            'params': params.copy(),
            'state_dict': best_state_dict,
        }

results_df = pd.DataFrame(results).sort_values('best_val_mse').reset_index(drop=True)
display(results_df)

Trial 01/16 | {'c_hidden': 16, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.001}
  Epoch 01 | train loss 6.404091 | val MSE 16.461942
  Epoch 02 | train loss 3.313128 | val MSE 14.951608
  Epoch 03 | train loss 2.386500 | val MSE 11.500353
  Epoch 04 | train loss 1.947279 | val MSE 10.060456
  Epoch 05 | train loss 1.701948 | val MSE 8.812874
  Epoch 06 | train loss 1.555118 | val MSE 7.936783
  Epoch 07 | train loss 1.457079 | val MSE 7.260421
  Epoch 08 | train loss 1.388616 | val MSE 7.092508
Trial 02/16 | {'c_hidden': 16, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.0001}
  Epoch 01 | train loss 9.826434 | val MSE 26.178739
  Epoch 02 | train loss 8.131211 | val MSE 23.360923
  Epoch 03 | train loss 7.141796 | val MSE 22.052595
  Epoch 04 | train loss 6.476836 | val MSE 20.939574
  Epoch 05 | train loss 5.980423 | val MSE 20.139073
  Epoch 06 | train loss 5.577186 | val MSE 19.494670
  Epoch 07 | train loss 5.203889 | val MSE 18.800304
  Epoch 08 | train loss 4.888099 | val MSE 18.230745
Trial 0

,c_hidden,num_heads,alpha,lr,best_val_mse
0,16,2,0.2,0.0010,7.092508
1,16,2,0.4,0.0010,7.725531
2,16,4,0.2,0.0010,8.231526
3,16,4,0.4,0.0010,8.519887
4,32,4,0.4,0.0010,10.683373
5,32,4,0.2,0.0010,11.652627
6,16,2,0.2,0.0001,18.230745
7,16,2,0.4,0.0001,18.903312
8,16,4,0.2,0.0001,21.347026
9,16,4,0.4,0.0001,22.471787


In [11]:
# reload best model
best_model = GAT(
    c_in=input_dim,
    c_hidden=best['params']['c_hidden'],
    c_out=1,
    num_relations=num_relations,
    num_heads=best['params']['num_heads'],
    alpha=best['params']['alpha']
)
best_model.load_state_dict(best['state_dict'])

# get predictions
pred_test = predict_GAT(best_model, test_loader, adj_matrix)
pred_test_np = pred_test.numpy()

# compute metrics
metrics = compute_metrics(y_test, pred_test_np)
print("Best params:", best['params'])
print("Test metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

Best params: {'c_hidden': 16, 'num_heads': 2, 'alpha': 0.2, 'lr': 0.001}
Test metrics:
  accuracy: 0.5009
  f1: 0.5003
  mcc: 0.0023
  return_ratio: 0.8484
  sharpe: 0.0698
  MSE: 2.4110


In [12]:
import os

save_dir = '../Best_model'
os.makedirs(save_dir, exist_ok=True)

torch.save({
    'params': best['params'],
    'state_dict': best['state_dict'],
    'metrics': metrics,
}, os.path.join(save_dir, 'GAT_best_model.pth'))

print("Model saved to ../Best_model/GAT_best_model.pth")

Model saved to ../Best_model/GAT_best_model.pth
